In [ ]:
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [ ]:
file_path = "../data/raw/Component_One_Core_Feasibility.csv"

df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)

Dataset shape: (1200, 25)


In [ ]:
y = df["feasibility_label"]

X = df.drop(columns=[
    "feasibility_label",
    "case_id",
    "business_description"
])

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (1200, 22)
y shape: (1200,)


In [ ]:
categorical_features = [
    "business_stage",
    "business_category",
    "district",
    "province",
    "location_type",
    "proposed_action",
    "competition_level"
]

numerical_features = [
    "available_capital_lkr",
    "loan_amount_lkr",
    "monthly_budget_lkr",
    "initial_inventory_cost_lkr",
    "expected_price_lkr",
    "expected_customers_per_day",
    "customer_demand_score",
    "expected_operating_days_per_month",
    "entrepreneur_experience_years",
    "location_suitability_score",
    "available_staff_count",
    "required_staff_count",
    "available_equipment_score",
    "required_equipment_score",
    "supplier_availability_score"
]

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numerical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (840, 22)
Validation: (180, 22)
Testing: (180, 22)


In [ ]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            random_state=42
        ))
    ]
)

In [ ]:
logistic_model.fit(X_train, y_train)

print("Logistic Regression training completed.")

Logistic Regression training completed.


In [ ]:
y_val_pred = logistic_model.predict(X_val)

print("Validation predictions generated.")

Validation predictions generated.


In [ ]:
accuracy = accuracy_score(y_val, y_val_pred)

precision = precision_score(
    y_val,
    y_val_pred,
    average="weighted",
    zero_division=0
)

recall = recall_score(
    y_val,
    y_val_pred,
    average="weighted",
    zero_division=0
)

f1 = f1_score(
    y_val,
    y_val_pred,
    average="weighted",
    zero_division=0
)

print("Validation Performance")
print("----------------------")
print("Accuracy :", round(accuracy, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1 Score :", round(f1, 4))

Validation Performance
----------------------
Accuracy : 0.7444
Precision: 0.7395
Recall   : 0.7444
F1 Score : 0.7353


In [ ]:
print(
    classification_report(
        y_val,
        y_val_pred,
        zero_division=0
    )
)

                        precision    recall  f1-score   support

Conditionally Feasible       0.76      0.84      0.80       102
              Feasible       0.73      0.74      0.74        47
            Infeasible       0.68      0.42      0.52        31

              accuracy                           0.74       180
             macro avg       0.72      0.67      0.69       180
          weighted avg       0.74      0.74      0.74       180



In [ ]:
cm = confusion_matrix(
    y_val,
    y_val_pred,
    labels=[
        "Infeasible",
        "Conditionally Feasible",
        "Feasible"
    ]
)

print(cm)

[[13 15  3]
 [ 6 86 10]
 [ 0 12 35]]


In [ ]:
random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            class_weight="balanced"
        ))
    ]
)

In [ ]:
random_forest_model.fit(X_train, y_train)

print("Random Forest training completed.")

Random Forest training completed.


In [ ]:
y_val_pred_rf = random_forest_model.predict(X_val)

accuracy_rf = accuracy_score(y_val, y_val_pred_rf)

precision_rf = precision_score(
    y_val,
    y_val_pred_rf,
    average="weighted",
    zero_division=0
)

recall_rf = recall_score(
    y_val,
    y_val_pred_rf,
    average="weighted",
    zero_division=0
)

f1_rf = f1_score(
    y_val,
    y_val_pred_rf,
    average="weighted",
    zero_division=0
)

print("Random Forest Validation Performance")
print("------------------------------------")
print("Accuracy :", round(accuracy_rf, 4))
print("Precision:", round(precision_rf, 4))
print("Recall   :", round(recall_rf, 4))
print("F1 Score :", round(f1_rf, 4))

Random Forest Validation Performance
------------------------------------
Accuracy : 0.8611
Precision: 0.8686
Recall   : 0.8611
F1 Score : 0.8618


In [ ]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy,
        accuracy_rf
    ],
    "Precision": [
        precision,
        precision_rf
    ],
    "Recall": [
        recall,
        recall_rf
    ],
    "F1 Score": [
        f1,
        f1_rf
    ]
})

results

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.744444,0.739498,0.744444,0.735287
1,Random Forest,0.861111,0.868640,0.861111,0.861800


In [ ]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Random Forest"
    ],
    "Accuracy": [
        accuracy,
        accuracy_rf
    ],
    "Precision": [
        precision,
        precision_rf
    ],
    "Recall": [
        recall,
        recall_rf
    ],
    "F1 Score": [
        f1,
        f1_rf
    ]
})

results

,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.744444,0.739498,0.744444,0.735287
1,Random Forest,0.861111,0.868640,0.861111,0.861800
